In [ ]:
import math
import numpy as np 
import matplotlib.pyplot as plt


In [ ]:
# Enable automatic display of all expressions that is typed at last of each cell

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
%matplotlib inline

In [ ]:
def f(x):
    return 3*x**2-4*x+5

In [ ]:
xs = np.arange(-5,5,0.25) 
ys = f(xs)
plt.plot(xs,ys)

In [ ]:
h =0.0001
x=3 
print((f(x+h)-f(x))/h) # if you incrase x by a small value h, how much does it alter f(x)? is given by this formula 

In [ ]:
#little complex example 
h = 0.0001 # closer to 0 value 

a= 2.0
b= -3.0
c=10.0
d = a*b + c 
print(d)


In [ ]:
#bump up a by a very small value 
#since b is negative and a is multiplied by b 
#incase you increase a, b is negative - so overall it adds lesser to d 
#hence you get the value -3.0000xx


h = 0.0001 # closer to 0 value 

a= 2.0
b= -3.0
c=10.0
d1 = a*b + c 
a+=h
d2= a*b + c 
print('d1 is',d1)
print('d2 is', d2)
print('slope is', ((d2-d1)/h))

print('a has a negative influence on d with a strength of 3.')


In [ ]:
#bump up a by b very small value 
#since a is negative and b is multiplied by a 
#incase you increase b, a is positive - so overall it adds more to d 
#hence you get the value 2.000


h = 0.0001 # closer to 0 value 

a= 2.0
b= -3.0
c=10.0
d1 = a*b + c 
b+=h
d2= a*b + c 
print('d1 is',d1)
print('d2 is', d2)
print('slope is', ((d2-d1)/h))

print('Slope = 2 means:If I increase b by a very small amount, the value of d increases twice as much as the increase in b.')

#In other words:

#For every +1 change in b, d changes by +2.
#b has a sensitivity of 2 with respect to d.

In [ ]:

h = 0.0001 # closer to 0 value 

a= 2.0
b= -3.0
c=10.0
d1 = a*b + c 
c+=h
d2= a*b + c 
print('d1 is',d1)
print('d2 is', d2)
print('slope is', ((d2-d1)/h))


In [ ]:
"""
FINAL SUMMARY: Understanding Derivatives & Slopes (Numerical Differentiation)

This block is purely for learning purposes.

What does the derivative (slope) mathematically mean?
-------------------------------------------------------
The derivative of a function f with respect to x is defined as:

    derivative/slope = (f(x + h) - f(x)) / h     where h is very close to 0

In simple terms:
- We have a function that depends on several variables.
- We slightly bump up one variable by a tiny value 'h'.
- how much does the function fluctuate ? 
- This tells us: "how sensitive is the function with regards to that particular bumped up variable ?"

This sensitivity is called the **slope** or **derivative** (gradient).

Example Function:
----------------
d = a * b + c

Here, a, b, and c are the contributors (inputs) to the function d.

Numerical Example:
-----------------
h = 0.0001          # very small value
a = 2.0
b = -3.0
c = 10.0

d1 = a * b + c      # = 4.0

# Bump up 'b' slightly
b += h
d2 = a * b + c      # ≈ 4.0002

slope = (d2 - d1) / h   # Result ≈ 2.0

What does Slope = 2 mean?
-------------------------
→ For every +1 increase in b, the value of d increases by +2.

→ b has a sensitivity (influence) of 2 with respect to d.

→ This number 2 is the derivative (∂d/∂b) of d with respect to b.

Real-life Analogy:
------------------
Think of 'd' as your final score.
Think of 'b' as one of the inputs.

If you increase b a little bit, your final score increases **twice as fast**.
Even though b is negative (-3), making it slightly less negative still improves d at a rate of 2.

This concept is fundamental in neural networks and backpropagation.
"""



In [ ]:
 #moving to neural networks 
#it will have massive mathematical expressions
#create a class that can hold this  
#example expression being : d1 = a*b + c 

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0 #initially gradient is 0 which means changing this variable does not change L/does not impact LOSS function
        self._backward = lambda : None                                  
        self._prev=set(_children)                     
        self._op = _op
        self.label = label 
      
        
    def __repr__(self):
        return f"Value(data={self.data})"

 
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data+other.data, (self, other), '+')

        def _backward(): # so why is this backward function inside the operator overloaders ?? _backward has to propagate the gradients to the values that created it- then each Value node should save a lit of children and change he grads of the chidlren and code gets real messy .
            #__add__ function knows what variables are being added - in other terms the child nodes which actually make up this node, so the gradiesnt have to be propagated to its children 
            self.grad += 1*out.grad  #chain rule (local derivative * global derivative ) -> here local derivative is 1 because its a + sign , global derivative is out.derivative 
            other.grad += 1*out.grad  #chain rule  (local derivative * global derivative )
        out._backward = _backward 
        return out

    def __radd__(self, other):  # other + self, e.g. 0 + Value(...)
        return self + other

        

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other ), '*')

        def _backward():
            self.grad += other.data * out.grad #chain rule (local derivative * global derivative ) -> here local derivative is other.grad , global derivative is out.grad  
            other.grad += self.data * out.grad #chain rule (local derivative * global derivative ) -> here local derivative is other.grad , global derivative is out.grad  
        out._backward =_backward 
        return out

    def __rmul__(self, other): #rmul is a falling back fuction if multiply does not work . for eg  we are trying to do 2*a--> mul will try and fail because self is not a Value object -> so it wil interchange to a,2 and send to rmul--> so we have to define that so that it does that 
        return self*other

    def exp(self):
        x =self.data
        out = Value(math.exp(x), (self,), 'exp')

        def _backward():
            self.grad = out.data * out.grad  #excercise 

        out._backward = _backward #what does this line mean ? doubt to do 
        return out 

    def __pow__(self, other ):
        assert isinstance(other, (int, float ))
        out = Value(self.data ** other, (self,), 'pow')   # only self is a child because other is considered a constant and not something you compute gradients against 

        def _backward():
            self.grad = (other)* (self.data ** (other-1 )) * out.grad #--> #local derivative( (other.data)* (self.data ** other.data-1) ) * global derivative is out.grad

        out._backward = _backward
        return out 

    def __neg__(self): #-other 
        return self*(-1)

    def __sub__(self, other): # self-other 
        return self + (-other)            

    def tanh(self):
        x=self.data
        o=((math.exp(2*x)-1)/(math.exp(2*x)+1))
        out = Value(o, (self,),'tanh')

        def _backward():
            self.grad += (1-o**2) * out.grad # out.grad is chai ned with the local derivative into self.grad 
        
        out._backward = _backward 
        return out

        #to implement division use this ----> #a/b = a*(1/b) = a*(b**-1)
    def __truediv__(self, other):
        return self *(other **-1)
        

    def backward(self):

        # topological order all of the children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # go one variable at a time and apply the chain rule to get its gradient
        self.grad = 1
        for v in reversed(topo):
            v._backward()



In [ ]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label,n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

In [ ]:
#learning block --> delete later 
# Test rendering with explicit render
# TRY OUT Value class and graoh vendoring for  a random equation and see the graphn   
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a*b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value(-2.0, label='f')
L = d * f; L.label = 'L'
L


dot = draw_dot(L)

# Method 1: Try to display
dot

# Method 2: Save as PNG (this should create the file)
try:
    dot.render('micrograd_graph', format='png', cleanup=True, view=False)
    print("Graph saved as micrograd_graph.png")
except Exception as e:
    print("Error:", e)

In [ ]:
 # the above diagram can be visualized as .. 
#L is the loss function. 
#we are going to find derivcative of L with respect to the weights of the neural net vis back propagation
#e are going to find the derivative of L with respect to the weights of data and not the data itself. data is going ot be fixed. weights are going to change 

In [ ]:
plt.plot(np.arange(-4,5,0.3), np.tanh(np.arange(-4,5,0.3)));plt.grid();

In [ ]:

# setting up of nodes - SETTING UP OF INPUT NODES TO MIMIC NEURON STRUCTURE 
#inputs 
x1=Value(2.0,label='x1')
x2=Value(0,label='x2')

#weights
w1 =Value(-3.0,label='w1')
w2=Value(1.0,label='w2')

#bias 
b=Value(6.8813735870195432, label='b')


x1w1=x1*w1; x1w1.label='x1w1'
x2w2=x2*w2; x2w2.label='x2w2'

#cell body function 
x1w1x2w2=x1w1+x2w2; x1w1x2w2.label='x1w1 + x2w1'
n= x1w1x2w2+b;n.label='n'

o=n.tanh();o.label='o'




In [ ]:
#learning block --> delete later 
#MANUAL BACK PROPAGATION THROUGH NEURON 
# step 1 : find gradient of o node- last node - its fixed - its going to be 1
o.grad=1.0;

"""@step 2 : find gradient of node n 
 we know o = tanh(n)
do/dn ---> to find the gradient of n node 
new formula for today-> differentiate tanh 
formula: 1-tanh(x)**2 ps: ** means square 

applying to our tanh 
o=tanh(n)
do/dn = 1-tanh(n)**2
do/dn = 1- o**2*/

 """

n.grad = 1 - (o.data **2);

"""step 3 : find gradient of nodes x1w1 + x2w1 and b , since the operation is 1, the gradient is just passed on from the next node into this without any changes """

x1w1x2w2.grad=n.grad
b.grad = n.grad

x1w1.grad = n.grad
x2w2.grad=n.grad

"""step 3 : find gradient of nodes X1 , W1, X2, W2  

w1, w2 are the weights that we care about. need to understand how it incfluences the final function so that we can adjust w1, w2 to maximum our final value

the final function we have chosen here is a tanh, but it doesnt have to do.. it can be tanh or any arbitrarily simple function , only thing is you should be able to calculate the local derivative 
of each weight with respect to the final function that you choose. just know how the variables can influence the final function 

now trick to find derivatives of w1 and w2 -with operation * being the local operation 

usually in terms of multiplication 

for eg: derivative of w2  = x2.data * x2w2.grad 
        derivative of x2 = w2.data * x2w2.grad 

        derivative of w1 = x1.data * x1w1.grad
        derivative of x1 = w1.data * x1w1.grad 
"""

x1.grad = w1.data * x1w1.grad
w1.grad = x1.data * x1w1.grad

x2.grad = w2.data * x2w2.grad
w2.grad = x2.data * x2w2.grad

"""
another obervation is :


w2 is always multiplied with x2 which is 0, so the influence of w2 should be none - hence w2.grad = 0 makes total sense 
"""









In [ ]:
#learning block --> delete later - 
# back propagation via code May 30 
o.grad = 1 
o._backward() 
n._backward()
b._backward()
x1w1x2w2._backward()
x1w1._backward()
x2w2._backward()
draw_dot(o)

In [ ]:
#final BACKWARD function added - keep this cell block
#MAY 31 

o.backward() 

draw_dot(o)

#previously we called back propagation on each node separately, but if you want to back propagate throughout the expression starting from the right most node 
# we use something called topological sort 
#topological sort is nothing but depth first search - dint go into detail of this, 
#intuition is - if we have to calculate _backward on any node, we cannot do it until everything after it is completed - everything after it - that depends on it is completed only then you can compute the _backward on current 
# now put this back in Value class itself AND THEN TRY o.backward() -> should give the same output as previous cell where backward is manually called 

"""def backward(self):

    # topological order all of the children in the graph
    topo = []
    visited = set()
    def build_topo(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:
                build_topo(child)
            topo.append(v)
    build_topo(self)

    # go one variable at a time and apply the chain rule to get its gradient
    self.grad = 1
    for v in reversed(topo):
        v._backward()"""



In [ ]:
#fixed a bug that appears on this specific use case 
# b.backward propagates the gradient to a twice because a+a and it overwrites 
# so in backward function make a change and deposit those gradients self.grad+=self.grad 

a= Value(3,label='a')
b=a+a
b.backward()
draw_dot(b)

In [ ]:
# setting up of nodes - SETTING UP OF INPUT NODES TO MIMIC NEURON STRUCTURE 
#inputs 
x1=Value(2.0,label='x1')
x2=Value(0,label='x2')

#weights
w1 =Value(-3.0,label='w1')
w2=Value(1.0,label='w2')

#bias 
b=Value(6.8813735870195432, label='b')


x1w1=x1*w1; x1w1.label='x1w1'
x2w2=x2*w2; x2w2.label='x2w2'

#cell body function 
x1w1x2w2=x1w1+x2w2; x1w1x2w2.label='x1w1 + x2w1'
n= x1w1x2w2+b;n.label='n'

o=n.tanh();o.label='o'


In [ ]:
o.backward()
draw_dot(o)

In [ ]:
# setting up of nodes - SETTING UP OF INPUT NODES TO MIMIC NEURON STRUCTURE 
#inputs 
x1=Value(2.0,label='x1')
x2=Value(0,label='x2')

#weights
w1 =Value(-3.0,label='w1')
w2=Value(1.0,label='w2')

#bias 
b=Value(6.8813735870195432, label='b')


x1w1=x1*w1; x1w1.label='x1w1'
x2w2=x2*w2; x2w2.label='x2w2'

#cell body function 
x1w1x2w2=x1w1+x2w2; x1w1x2w2.label='x1w1 + x2w1'
n= x1w1x2w2+b;n.label='n'

temp = (2*n).exp()

a = temp-1
b = temp+1
print(a)
print(b)
tanhval = a/b
print(tanhval)
tanhval.backward()
draw_dot(tanhval)

In [ ]:
#June 27 

#to do above same with pytorch apis 
#implementing the neuron structure with 
#tensors are n-dimensional arrays of scalars 
#doubt means  float64 - double precision 
#since auot
x1 = torch.Tensor([2.0]).double() ; x1.requires_grad =True 
x2 = torch.Tensor([0]).double() ; x2.requires_grad =True 
w1 = torch.Tensor([-3.0]).double() ; w1.requires_grad =True 
w2 = torch.Tensor([1.0]).double() ; w2.requires_grad =True 



b= torch.Tensor([6.8813735870195432])

n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

o.backward()





In [ ]:
#June 27 
#just like in auotgrad, pytorch also has data , grad 

x1.data.item() # .item strips the tensor out and just gives the value 

In [ ]:
#June 27 
x1.grad

In [ ]:
x1.grad.item() # .item strips the tensor out and just gives the value 

In [ ]:
o.data.item() #forward pass 


In [ ]:
print('x1.grad =', x1.grad.item())
print('x2.grad =', x2.grad.item())
print('w1.grad =', w1.grad.item())
print('w2.grad =', w2.grad.item())

#compare with our autograd engine - its the same 

In [ ]:
#June 27 
import random

class Neuron:
    def __init__(self, nin):
        self.w = [ Value(random.uniform(-1,1)) for _ in range(nin)] # you have a list of weights - each weight is a Value object - the number of weights is equal to the number of inputs
        self.b = Value(random.uniform(-1,1))#happiness bias of that neuron 


    def __call__(self, x): #x= [2.0, 3.0]
        act = sum((xi*wi for xi,wi in zip(self.w, x)),self.b)
        out = act.tanh()
        return out

    def parameters(self): #have  amechanism to strip out the weights and the bias to this neuron so that it can be changed altered to mimise the loss in the future 
        return self.w +[self.b]


class Layer: #define a layer of neurons - each neuron has nin inputs and there are nout neurons in this layer 
    def __init__(self, nin, nouts):
        self.neurons = [Neuron(nin) for _ in range(nouts)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        # params = []
        # for neuron in self.neurons :
        #      param = neuron.parameters()
        #      params.extend(param)

        # return params 
        return [p for neuron in self.neurons for p in neuron.parameters()]
            
            
        
        
class MLP:
    def __init__(self, nin, nouts): # but here nouts will be a list, each item in the list gives the size of that layer . if nouts is [4,4,1] - first layer has 4 neurons/outputs, second layer has 4 neurons, third layer has one final output
        # to define the set of layers you need a mathetimatical representation 
        #nin = 2 means each neuron will have 2 inputs 
        #to understand the boundary of each layer or access each layer 
        #[2,4,4,1] array would be very useful 
        # because first layer can be defined as Layer(2,4)
        #second layer can be defined as Layer (4,4)
        #third layer will be (4,1)
        #hence this operation 
        sz = [nin] + nouts  # this concatenates the list [2,4,4,1] will be size
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):# x is the input to the first layer 
        for layer in self.layers:
          x = layer(x)
        return x

        #lets say number if inputs to each neuron is 2 = nin
        #[4,4,1] is the size of each layer = nouts 
        #so in __init__ ML creates  a list of Layers using the line self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
        #so now we have [Layer(2,3), Layer(4,4), Layer(4,1)]
        
        # you pass x = [2.0,3.0,4.0] as the input to the first  layer right 
        #how does it propagate to the next layers 
        #how does input of one layer pass to the next layer ???
# iteration 1:
# layer = self.layers[0]  -> this is Layer(2,4)
# x = layer(x)  -> sending original input [2.0, 3.0] into layer 1
# layer 1 has 4 neurons, so it returns 4 outputs, something like [0.5, -0.2, 0.9, 0.1]
# x gets overwritten now, x is no longer [2.0, 3.0], x is now [0.5, -0.2, 0.9, 0.1]

# iteration 2:
# layer = self.layers[1]  -> this is Layer(4,4)
# x = layer(x)  -> sending [0.5, -0.2, 0.9, 0.1] (output of layer 1) into layer 2
# this works because layer 2 expects 4 inputs, and we are giving it exactly 4 numbers
# layer 2 also has 4 neurons, so it again returns 4 outputs, say [0.3, 0.7, -0.1, 0.6]
# x gets overwritten again, x is now [0.3, 0.7, -0.1, 0.6]

# iteration 3:
# layer = self.layers[2]  -> this is Layer(4,1)
# x = layer(x)  -> sending [0.3, 0.7, -0.1, 0.6] (output of layer 2) into layer 3
# layer 3 expects 4 inputs, matches fine
# layer 3 has only 1 neuron, so it returns just 1 final output, say 0.85
# x gets overwritten one last time, x is now just 0.85

# after the loop ends, return x
# this 0.85 is the final output of the whole MLP
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

            
    
    
        


        

In [ ]:

x= [2.0, 3.0]
n = Neuron(2) # one single neuron takes 2 inputs and activates and sends out 1 output 
n(x)

In [ ]:
x= [2.0, 3.0]
n = Layer(2, 4) # each neuron is 2 dimensional which means it takes 2 inputs, and there are 4 neurons in total 
#each neuron takes those 2 inputs, activates and returns a final value 
#since there are 4 neurons in this layer, the output will have 4 values 
n(x)

In [ ]:
x= [2.0, 3.0]
nin=2# number of inputs to each neuron in layer 1  
nouts = [4,4,1] #number of neurons in each layer of the MLP - based on nin and nout, layers would have been defined in this statement 
n = MLP(nin, nouts) #explanation of how MLP feed fprward networks work is given
n(x)


In [ ]:
# finaly test yourself by understanding whats the input dimension, output dimension etc., of how to define Nueron, Layer, MLP 
x= [2.0, 3.0]
n =  Neuron(2)
n(x)

l=Layer(2, 3)
l(x)

mlp =MLP(2, [4,4,1])
mlp(x)

In [ ]:
#check parameters
x= [2.0, 3.0, 5.0]
mlp =MLP(3, [4,4,1])
mlp(x)
len(mlp.parameters()) 

In [ ]:
#working on nueral nets for a sample data set 
n =  MLP(3, [4,4,1])
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets
ypred = [n(x) for x in xs ]
loss = sum((yout-ygt)**2 for ygt, yout in zip(ys, ypred))
loss


In [ ]:
loss.backward() #fills up the grad data for each neuron in the network- it will give information on how it influences the loss function

In [ ]:
n.layers[0].neurons[0].w[0].grad

In [ ]:
n.layers[0].neurons[0].w[0].data

In [ ]:
# little bit of notes  on what we understand from n.layers[0].neurons[0].w[0].grad

#loss. backward filled all the grads of neurons in this mlp with respect to the loss function 

# initially lets say the loss is Value(data=5.0148361360211435)

# since the loss is higher , lets check whats the status of the grads 

# -0.4874730142826441 is the gradient of a random neuron n.layers[0].neurons[0].w[0].grad

#a negative gradient means its influencing the loss function negatively 

#which means that particular neuron which has a negative gradient is in the opposite direction of the loss, it influences the loss oppositely  

#the same neuron's data n.layers[0].neurons[0].w[0].data - if it is increased- it will decrease the loss, or if it is decreased it will increase the loss - that's what a negative gradient means - it has an opposite influence 

#now our goal should be to nudge the weight's data in a way that the loss is decreased which means the data should be increased ( because negative gradient )


#p.data  += 0.01*p.grad -> if you use this to correct the weights , it wont work because grad is negative - hence p.data will be reduced - which will in turn increase the loss 

#hence we do p.data  += -0.01*p.grad  --> this will nudge the weight slighly increased (p.grad is negative and -0.01 is negative hence += on a positive number, increases the data  ) which means our loss will be reduced 


In [ ]:
#working on nueral nets for a sample data set 
n =  MLP(3, [4,4,1])
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

#gradient descent 

for _ in range(20):
    
    #forward pass 
    ypred = [n(x) for x in xs ]
    # find the loss
    loss = sum((yout-ygt)**2 for ygt, yout in zip(ys, ypred)) 

    # clear up the previous step's run's gradients before calculating grads on this pass's loss - dont keep the old grad values 
    for p in n.parameters():
        p.grad = 0.0
    #do backward pass and fill the gradients of all neurons of mlp to understand its influence on the loss
    loss.backward()

    print('step : ', _, loss.data)

    #update the weights with the information we have on p.grad of each parameter because that tells the influence it has on the loss function
    for p in n.parameters():
        p.data+= -0.05*p.grad

ypred

    #do the same thing 20 times 
    #you final ypred should be closer to ys 
    #your final loss should be much lesser 
    # this repated process of 
#-> forward pass 
#-> find the loss 
#->adjust the weights 
#-> repeat for 20 times 


    
    
    